In [ ]:
!pip install librosa==0.10.2

In [ ]:
import os
import random
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')



In [ ]:
DATA_ROOT = "/kaggle/input/datasets/imsparsh/fma-free-music-archive-small-medium/fma_medium/fma_medium"
META_ROOT = "/kaggle/input/datasets/imsparsh/fma-free-music-archive-small-medium/fma_metadata"


In [ ]:
tracks = pd.read_csv(
    f"{META_ROOT}/tracks.csv",
    index_col=0,
    header=[0,1]
)

genres = pd.read_csv(
    f"{META_ROOT}/genres.csv",
    index_col=0
)

print("Total tracks:", len(tracks))


In [ ]:
genre_map = {
0:"Classical",
1:"Electronic",
2:"Experimental",
3:"Folk",
4:"Hip-Hop",
5:"Instrumental",
6:"International",
7:"Old-Time",
8:"Pop",
9:"Rock"
}

valid_genres = set(genre_map.values())


In [ ]:
tracks_subset = tracks[tracks[("set","subset")] == "medium"]

print("Medium tracks:", len(tracks_subset))


In [ ]:
def track_to_path(track_id):

    tid = str(track_id).zfill(6)

    folder = tid[:3]

    return f"{DATA_ROOT}/{folder}/{tid}.mp3"


In [ ]:
rows = []

for tid,row in tracks_subset.iterrows():

    genre = row[("track","genre_top")]

    if genre in valid_genres:

        rows.append({
            "track_id": tid,
            "file_path": track_to_path(tid),
            "genre": genre
        })

df = pd.DataFrame(rows)

print("Usable tracks:", len(df))


In [ ]:
label_to_index = {v:k for k,v in genre_map.items()}

df["genre_index"] = df["genre"].map(label_to_index)

df["genre"].value_counts()


In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["genre_index"],
    random_state=42
)

print("Train:", len(train_df))
print("Val:", len(val_df))


In [ ]:
CACHE_DIR = "/kaggle/working/mel_cache"

os.makedirs(CACHE_DIR, exist_ok=True)

print("Cache dir:", CACHE_DIR)


In [ ]:
def audio_to_mel(path):

    try:

        y, sr = librosa.load(path, sr=22050)

        segment_samples = 22050 * 10

        if len(y) > segment_samples:
            y = y[:segment_samples]
        else:
            y = np.pad(y, (0, segment_samples-len(y)))

        mel = librosa.feature.melspectrogram(
            y=y,
            sr=22050,
            n_mels=128,
            n_fft=2048,
            hop_length=512
        )

        mel = librosa.power_to_db(mel, ref=np.max)

        mel = (mel - mel.mean()) / (mel.std()+1e-6)

        if mel.shape[1] < 431:
            mel = np.pad(mel, ((0,0),(0,431-mel.shape[1])))

        mel = mel[:, :431]

        return mel[..., np.newaxis]

    except:
        return None


In [ ]:
X_train = []
y_train = []

print("Generating training mels...")

for _,row in tqdm(train_df.iterrows(), total=len(train_df)):

    mel = audio_to_mel(row.file_path)

    if mel is None:
        continue

    X_train.append(mel)
    y_train.append(row.genre_index)

X_train = np.array(X_train)
y_train = np.array(y_train)

print(X_train.shape)


In [ ]:
X_val = []
y_val = []

print("Generating validation mels...")

for _,row in tqdm(val_df.iterrows(), total=len(val_df)):

    mel = audio_to_mel(row.file_path)

    if mel is None:
        continue

    X_val.append(mel)
    y_val.append(row.genre_index)

X_val = np.array(X_val)
y_val = np.array(y_val)

print(X_val.shape)


In [ ]:
np.save(f"{CACHE_DIR}/X_train.npy", X_train)
np.save(f"{CACHE_DIR}/y_train.npy", y_train)

np.save(f"{CACHE_DIR}/X_val.npy", X_val)
np.save(f"{CACHE_DIR}/y_val.npy", y_val)

print("Dataset cached")


In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

print(class_weights)


In [ ]:
from tensorflow.keras import layers

def cnn_block(x, filters):

    x = layers.Conv2D(filters,3,padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters,3,padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.MaxPool2D((2,2))(x)

    return x


def transformer_block(x):

    attn = layers.MultiHeadAttention(
        num_heads=4,
        key_dim=x.shape[-1]
    )(x,x)

    x = layers.Add()([x,attn])
    x = layers.LayerNormalization()(x)

    ff = layers.Dense(256,activation="relu")(x)
    ff = layers.Dense(x.shape[-1])(ff)

    x = layers.Add()([x,ff])
    x = layers.LayerNormalization()(x)

    return x


def build_model():

    inputs = layers.Input((128,431,1))

    x = cnn_block(inputs,32)
    x = cnn_block(x,64)
    x = cnn_block(x,128)
    x = cnn_block(x,256)

    x = layers.Permute((2,1,3))(x)

    x = layers.Reshape((-1, x.shape[2]*x.shape[3]))(x)

    x = transformer_block(x)
    x = transformer_block(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(256,activation="relu")(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(10,activation="softmax")(x)

    model = tf.keras.Model(inputs,outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(3e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


model = build_model()

model.summary()


In [ ]:
callbacks = [

    tf.keras.callbacks.EarlyStopping(
        patience=8,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        patience=4,
        factor=0.5
    )

]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val,y_val),
    epochs=50,
    batch_size=8,
    callbacks=callbacks,
    class_weight=class_weights
)


In [ ]:
model.save("/kaggle/working/best_genre_transformer.keras")
